# Vietnamese Receipt EDA

## §0 Setup & Data Acquisition

In [ ]:
import os
import random
import hashlib
import json
import yaml
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

# --- Project-local cache dirs (gitignored) ---
PROJECT_ROOT = Path(".").resolve()
DATASETS_DIR = PROJECT_ROOT / "datasets"
os.environ["KAGGLEHUB_CACHE"] = str(DATASETS_DIR / "kagglehub")
os.environ["HF_HOME"] = str(DATASETS_DIR / "huggingface")
(DATASETS_DIR / "kagglehub").mkdir(parents=True, exist_ok=True)
(DATASETS_DIR / "huggingface").mkdir(parents=True, exist_ok=True)

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- Output dir (recreate from scratch each run) ---
OUT = Path("eda_outputs")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

# --- Download MC-OCR from Kaggle ---
# NOTE: VQA (`5CD-AI/Viet-OCR-VQA-flash2`) is deferred — gated access pending manual review.
import kagglehub
print("Downloading MC-OCR from Kaggle (cached under ./datasets/kagglehub/)...")
mcocr_path = Path(kagglehub.dataset_download("domixi1989/vietnamese-receipts-mc-ocr-2021"))
print(f"MC-OCR unpacked to: {mcocr_path}")

# Reproducibility hash for MC-OCR (Kaggle has no clean revision)
def tree_hash(path):
    h = hashlib.sha256()
    for p in sorted(path.rglob("*")):
        if p.is_file():
            h.update(str(p.relative_to(path)).encode())
            h.update(str(p.stat().st_size).encode())
    return h.hexdigest()
MCOCR_TREE_HASH = tree_hash(mcocr_path)
print(f"MC-OCR tree hash (size-based): {MCOCR_TREE_HASH[:16]}...")

# --- MC-OCR layout ---
# Confirmed via prior inspection of versions/17/ archive:
#   CSV:        mcocr_train_df.csv at the archive root
#   Image dir:  data0.7/data0.7/
#   Filename:   img_id column
mcocr_csv = mcocr_path / "mcocr_train_df.csv"
mcocr_img_dir = mcocr_path / "data0.7" / "data0.7"
print(f"MC-OCR CSV: {mcocr_csv}")
print(f"MC-OCR image dir: {mcocr_img_dir}")
mcocr_df = pd.read_csv(mcocr_csv)
print(f"MC-OCR CSV columns: {list(mcocr_df.columns)}")
print(f"MC-OCR CSV head:\n{mcocr_df.head(2).to_string()}")

def iter_mcocr():
    """Yield (example_id, image_path_str, annotation_dict)."""
    for _, row in mcocr_df.iterrows():
        name = str(row["img_id"])
        img_path = mcocr_img_dir / name
        if not img_path.exists():
            continue
        yield (name, str(img_path), row.to_dict())

# --- Materialize counts (cheap) ---
MCOCR_COUNT = sum(1 for _ in iter_mcocr())
print(f"\nMC-OCR examples reachable: {MCOCR_COUNT}")

## §1 Schema & Field Analysis → schema.json

In [ ]:
def annotation_pairs():
    """Yield (dataset_name, ann_dict) for MC-OCR. VQA half deferred (see scope amendment)."""
    for _id, _img, ann in iter_mcocr():
        yield ("mc_ocr", ann if isinstance(ann, dict) else dict(ann))

field_obs = {}
totals = Counter()

for ds, ann in annotation_pairs():
    totals[ds] += 1
    for k, v in ann.items():
        rec = field_obs.setdefault(k, {"datasets": set(), "present_per_ds": Counter(), "samples": []})
        rec["datasets"].add(ds)
        if v is not None and v != "" and v != [] and not (isinstance(v, float) and pd.isna(v)):
            rec["present_per_ds"][ds] += 1
            if len(rec["samples"]) < 5:
                rec["samples"].append(str(v)[:120])

def infer_type(samples):
    if not samples:
        return "string"
    if all(s.replace(",", "").replace(".", "").replace("-", "").replace(" ", "").isdigit()
           for s in samples if s):
        return "number"
    if any(("/" in s and len(s) <= 12) or ("-" in s and len(s) == 10) for s in samples):
        return "date"
    if any(s.lstrip().startswith("[") for s in samples):
        return "array"
    return "string"

fields = []
for name, rec in sorted(field_obs.items()):
    denom = sum(totals[d] for d in rec["datasets"])
    coverage = sum(rec["present_per_ds"].values()) / denom if denom else 0.0
    fields.append({
        "name": name,
        "type": infer_type(rec["samples"]),
        "source_datasets": sorted(rec["datasets"]),
        "coverage_rate": round(coverage, 3),
        "format_observations": rec["samples"][:5],
    })

schema = {"fields": fields, "totals": dict(totals)}
(OUT / "schema.json").write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"§1 wrote schema.json with {len(fields)} fields")
for f in fields[:10]:
    print(f"  {f['name']:30s} type={f['type']:7s} cov={f['coverage_rate']:.2f} ds={f['source_datasets']}")

## §2 Image Characteristics → training_caps.yaml (resolution)

In [ ]:
import matplotlib.pyplot as plt

def open_size(img_or_path):
    if hasattr(img_or_path, "size") and not isinstance(img_or_path, (str, Path)):
        return img_or_path.size  # PIL image -> (W, H)
    return Image.open(img_or_path).size

records = []
for _id, img_path, _ann in iter_mcocr():
    w, h = open_size(img_path)
    sz = Path(img_path).stat().st_size
    records.append({"dataset": "mc_ocr", "w": w, "h": h, "file_size": sz})

df_img = pd.DataFrame(records)
df_img["longest_side"] = df_img[["w", "h"]].max(axis=1)
df_img["aspect_ratio"] = df_img["h"] / df_img["w"]

print(f"§2 image stats over {len(df_img)} images:")
print(df_img.groupby("dataset")[["longest_side", "aspect_ratio"]].describe().to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_img["longest_side"].hist(bins=50, ax=axes[0]); axes[0].set_title("Longest side (px)")
df_img["aspect_ratio"].hist(bins=50, ax=axes[1]); axes[1].set_title("Aspect ratio (h/w)")
df_img["file_size"].dropna().hist(bins=50, ax=axes[2]); axes[2].set_title("File size (bytes)")
plt.tight_layout()
plt.show()

ls = df_img["longest_side"]
percentiles = {"p50": int(ls.quantile(0.50)), "p95": int(ls.quantile(0.95)), "p99": int(ls.quantile(0.99))}
max_res = min(1920, percentiles["p95"])
target_res = percentiles["p50"]

caps = {
    "max_resolution": int(max_res),
    "target_resolution": int(target_res),
    "longest_side_percentiles": percentiles,
    "n_images_analyzed": int(len(df_img)),
    "max_resolution_rule": "min(P95 of longest-side, 1920) — 1920 is the sanity bound for VRAM",
}
(OUT / "training_caps.yaml").write_text(yaml.safe_dump(caps, sort_keys=False), encoding="utf-8")
print(f"§2 wrote training_caps.yaml: max_resolution={max_res}, target_resolution={target_res}")

## §3 Text Characteristics → normalization_rules.yaml + training_caps.yaml (seq_length)

In [ ]:
import re

VN_DIACRITIC_CHARS = set(
    "àáảãạăằắẳẵặâầấẩẫậ"
    "èéẻẽẹêềếểễệ"
    "ìíỉĩị"
    "òóỏõọôồốổỗộơờớởỡợ"
    "ùúủũụưừứửữự"
    "ỳýỷỹỵ"
    "đ"
    "ÀÁẢÃẠĂẰẮẲẴẶÂẦẤẨẪẬ"
    "ÈÉẺẼẸÊỀẾỂỄỆ"
    "ÌÍỈĨỊ"
    "ÒÓỎÕỌÔỒỐỔỖỘƠỜỚỞỠỢ"
    "ÙÚỦŨỤƯỪỨỬỮỰ"
    "ỲÝỶỸỴ"
    "Đ"
)

# MC-OCR text fields (anno_polygons is coordinate data; img_id/anno_num/anno_image_quality are not text).
TEXT_FIELDS = ("anno_texts", "anno_labels")

def text_payloads():
    """Yield strings from the MC-OCR text fields only (skips polygon coordinate blobs)."""
    for _ds, ann in annotation_pairs():
        for k in TEXT_FIELDS:
            v = ann.get(k)
            if isinstance(v, str) and v:
                yield v

char_counter = Counter()
diac_chars = 0
total_chars = 0
for s in text_payloads():
    char_counter.update(s)
    diac_chars += sum(1 for c in s if c in VN_DIACRITIC_CHARS)
    total_chars += len(s)
diac_density = diac_chars / total_chars if total_chars else 0.0
print(f"§3 char stats: total_chars={total_chars}, diacritic_density={diac_density:.4f}, "
      f"unique_chars={len(char_counter)}")

NUMERIC_RE = re.compile(r"\d{1,3}(?:[.,\s]\d{3})*(?:[.,]\d+)?")
DATE_RE = re.compile(r"\b\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}\b")
CURRENCY_RE = re.compile(r"(đ|₫|VND|VNĐ|đồng)\b", re.IGNORECASE)

def pattern_signature(s):
    return "".join("D" if c.isdigit() else c for c in s)

numeric_patterns = Counter()
date_patterns = Counter()
currency_tokens = Counter()
for s in text_payloads():
    for m in NUMERIC_RE.findall(s):
        numeric_patterns[pattern_signature(m)] += 1
    for m in DATE_RE.findall(s):
        date_patterns[pattern_signature(m)] += 1
    for m in CURRENCY_RE.findall(s):
        currency_tokens[m.lower()] += 1

print("Top numeric patterns:", numeric_patterns.most_common(10))
print("Top date patterns:   ", date_patterns.most_common(10))
print("Currency tokens:     ", currency_tokens.most_common())

def get_tokenizer():
    try:
        from transformers import AutoTokenizer
        return AutoTokenizer.from_pretrained("Qwen/Qwen2-VL-2B-Instruct"), 1.0
    except Exception as e:
        print(f"Tokenizer unavailable ({e!r}); using whitespace token count × 1.3 safety multiplier")
        return None, 1.3

tok, safety = get_tokenizer()

def count_tokens(s):
    if tok is not None:
        return len(tok.encode(s, add_special_tokens=False))
    return int(len(s.split()) * safety)

def receipt_text(ann):
    """Concatenate the MC-OCR text fields for one receipt (skips coordinate/metadata fields)."""
    parts = [ann[k] for k in TEXT_FIELDS if isinstance(ann.get(k), str)]
    return "\n".join(parts)

token_counts = []
for _ds, ann in annotation_pairs():
    token_counts.append(count_tokens(receipt_text(ann)))
tc = pd.Series(token_counts)
p99 = int(tc.quantile(0.99))
max_seq_length = max(128, min(int(p99 * 1.1), 8192))
print(f"§3 token-count percentiles: p50={int(tc.quantile(0.5))}, "
      f"p95={int(tc.quantile(0.95))}, p99={p99}; max_seq_length={max_seq_length}")

norm = {
    "numeric": {
        "patterns": [{"pattern": p, "count": c} for p, c in numeric_patterns.most_common(20)],
        "canonical": "Strip thousands separators (. , space). Replace decimal comma with dot. Parse as float.",
    },
    "date": {
        "patterns": [{"pattern": p, "count": c} for p, c in date_patterns.most_common(20)],
        "canonical": "Parse to ISO 8601 (YYYY-MM-DD). Heuristic: if first part > 31, assume YYYY-MM-DD; else DD/MM/YYYY.",
    },
    "currency": {
        "patterns": [{"token": t, "count": c} for t, c in currency_tokens.most_common()],
        "canonical": "Strip currency suffix tokens (đ, ₫, VND, VNĐ, đồng) before numeric parse.",
    },
    "diacritic_density": round(diac_density, 4),
}
(OUT / "normalization_rules.yaml").write_text(
    yaml.safe_dump(norm, sort_keys=False, allow_unicode=True), encoding="utf-8"
)
print(f"§3 wrote normalization_rules.yaml")

caps_path = OUT / "training_caps.yaml"
caps = yaml.safe_load(caps_path.read_text(encoding="utf-8"))
caps["max_seq_length"] = max_seq_length
caps["token_count_percentiles"] = {
    "p50": int(tc.quantile(0.5)),
    "p95": int(tc.quantile(0.95)),
    "p99": p99,
}
caps_path.write_text(yaml.safe_dump(caps, sort_keys=False), encoding="utf-8")
print(f"§3 updated training_caps.yaml with max_seq_length={max_seq_length}")

## §4 Visual Quality (Automated) → augmentation.yaml

## §5 VQA Structure → prompt_templates.yaml

## §6 Cross-Dataset Comparison → training_strategy.yaml

## §7 Summary & Sanity Checks

In [ ]:
import os, json, yaml
from pathlib import Path

OUT = Path("eda_outputs")

def validate_section_0():
    """§0 produced MC-OCR iterator with non-zero count."""
    assert "MCOCR_COUNT" in globals(), "§0 did not run; MCOCR_COUNT undefined"
    assert MCOCR_COUNT > 0, "MC-OCR yielded zero examples"
    print(f"  §0 OK — MC-OCR: {MCOCR_COUNT}")

def validate_section_1():
    """§1 emits schema.json with non-empty fields list and valid coverage rates."""
    p = OUT / "schema.json"
    assert p.exists(), f"{p} missing"
    schema = json.loads(p.read_text(encoding="utf-8"))
    assert "fields" in schema, "schema.json has no 'fields' key"
    assert len(schema["fields"]) > 0, "schema.json fields list is empty"
    for f in schema["fields"]:
        for key in ("name", "type", "source_datasets", "coverage_rate", "format_observations"):
            assert key in f, f"field missing key '{key}': {f}"
        assert 0.0 <= f["coverage_rate"] <= 1.0, f"coverage_rate out of range: {f}"
        assert f["type"] in ("string", "number", "date", "array"), f"unknown type: {f}"
    print(f"  §1 OK — {len(schema['fields'])} fields cataloged")

def validate_section_2():
    """§2 emits training_caps.yaml with valid max_resolution and percentiles."""
    p = OUT / "training_caps.yaml"
    assert p.exists(), f"{p} missing"
    caps = yaml.safe_load(p.read_text(encoding="utf-8"))
    for key in ("max_resolution", "target_resolution", "longest_side_percentiles"):
        assert key in caps, f"training_caps.yaml missing key '{key}'"
    assert 256 <= caps["max_resolution"] <= 4096, f"max_resolution {caps['max_resolution']} out of [256, 4096]"
    assert 256 <= caps["target_resolution"] <= caps["max_resolution"], \
        f"target_resolution {caps['target_resolution']} not in [256, max_resolution]"
    assert "p50" in caps["longest_side_percentiles"], "percentiles missing p50"
    print(f"  §2 OK — max_resolution={caps['max_resolution']}, target={caps['target_resolution']}")

def validate_section_3():
    """§3 emits normalization_rules.yaml + appends max_seq_length to training_caps.yaml."""
    p_norm = OUT / "normalization_rules.yaml"
    p_caps = OUT / "training_caps.yaml"
    assert p_norm.exists(), f"{p_norm} missing"
    norm = yaml.safe_load(p_norm.read_text(encoding="utf-8"))
    for key in ("numeric", "date", "currency"):
        assert key in norm, f"normalization_rules.yaml missing '{key}'"
        assert "patterns" in norm[key] and "canonical" in norm[key], \
            f"normalization_rules.yaml '{key}' missing patterns/canonical"

    caps = yaml.safe_load(p_caps.read_text(encoding="utf-8"))
    assert "max_seq_length" in caps, "training_caps.yaml missing max_seq_length"
    assert 128 <= caps["max_seq_length"] <= 8192, f"max_seq_length out of [128, 8192]: {caps['max_seq_length']}"
    print(f"  §3 OK — max_seq_length={caps['max_seq_length']}, "
          f"{len(norm['numeric']['patterns'])} numeric patterns, "
          f"{len(norm['date']['patterns'])} date patterns")

def run_all_validations():
    print("Running sanity checks...")
    validate_section_0()
    validate_section_1()
    validate_section_2()
    validate_section_3()
    print("All sanity checks passed.")

run_all_validations()